<a href="https://colab.research.google.com/github/oswrik/suscripcion_banco/blob/main/notebooks/03_optimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Optimización con Optuna y comparación honesta (Producto 2)
En el Producto 1 ganó **LightGBM**. Ahora optimizamos sus hiperparámetros con Optuna y comprobamos **fold a fold** si logra superar a su versión original.

In [7]:
!pip install -q optuna

In [8]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Datos: Bank Marketing (UCI, id 222)
URL = "https://raw.githubusercontent.com/oswrik/suscripcion_banco/refs/heads/main/data/bank-full.csv"

NUMERICAS = ["edad", "moroso", "saldo", "hipoteca", "prestamo",
             "dia", "campana", "dias_previos", "contactos_previos"]
CATEGORICAS = ["trabajo", "estado_civil", "educacion", "contacto", "mes", "resultado_previo"]

def cargar_datos():
    df = pd.read_csv(URL, sep=";")
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={
        "age": "edad", "job": "trabajo", "marital": "estado_civil",
        "education": "educacion", "default": "moroso", "balance": "saldo",
        "housing": "hipoteca", "loan": "prestamo", "contact": "contacto",
        "day": "dia", "month": "mes", "campaign": "campana",
        "pdays": "dias_previos", "previous": "contactos_previos",
        "poutcome": "resultado_previo"})
    for col in ["moroso", "hipoteca", "prestamo"]:
        df[col] = (df[col].str.strip() == "yes").astype(int)
    df["suscribe"] = (df["y"].str.strip() == "yes").astype(int)
    return df[NUMERICAS + CATEGORICAS + ["suscribe"]]

df = cargar_datos()
X, y = df[NUMERICAS + CATEGORICAS], df["suscribe"]
print(df.shape, f"| suscriben: {y.mean():.1%}")

(45211, 16) | suscriben: 11.7%


In [9]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Misma división en todos los notebooks: el test se usa una sola vez, al final
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def preprocesador():
    num = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
    cat = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num, NUMERICAS), ("cat", cat, CATEGORICAS)])

In [10]:
import optuna
from lightgbm import LGBMClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    m = LGBMClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 100),
        subsample=trial.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        class_weight="balanced", random_state=42, verbose=-1, n_jobs=1)
    return cross_val_score(Pipeline([("prep", preprocesador()), ("modelo", m)]),
                           X_tr, y_tr, cv=cv, scoring="roc_auc").mean()

estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
estudio.optimize(objetivo, n_trials=30)
print(f"Mejor ROC-AUC (CV): {estudio.best_value:.4f}")
print("Mejores hiperparámetros:", estudio.best_params)

Mejor ROC-AUC (CV): 0.8014
Mejores hiperparámetros: {'n_estimators': 448, 'learning_rate': 0.010829830253261452, 'num_leaves': 41, 'min_child_samples': 43, 'subsample': 0.8217351462700548, 'colsample_bytree': 0.7558121832094163}


## ¿La mejora es real? Comparación fold a fold (5 folds × 5 repeticiones)

In [11]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from scipy.stats import wilcoxon

cv25 = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
original = Pipeline([("prep", preprocesador()),
                     ("modelo", LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1, n_jobs=1))])
logistica = Pipeline([("prep", preprocesador()),
                      ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced"))])
optimizado = Pipeline([("prep", preprocesador()),
                       ("modelo", LGBMClassifier(**estudio.best_params, subsample_freq=1, class_weight="balanced",
                                                 random_state=42, verbose=-1, n_jobs=1))])
a = cross_val_score(optimizado, X_tr, y_tr, cv=cv25, scoring="roc_auc")
b = cross_val_score(original, X_tr, y_tr, cv=cv25, scoring="roc_auc")
c = cross_val_score(logistica, X_tr, y_tr, cv=cv25, scoring="roc_auc")
print(f"LightGBM optimizado:  {a.mean():.4f} ± {a.std():.4f}")
print(f"LightGBM original:    {b.mean():.4f} ± {b.std():.4f}")
print(f"Regresión logística:  {c.mean():.4f} ± {c.std():.4f}")
for nombre, otro in [("LightGBM original", b), ("Regresión logística", c)]:
    dif = a - otro
    print(f"Optimizado vs. {nombre}: gana en {(dif > 0).sum()} de {len(dif)} folds · Wilcoxon p = {wilcoxon(dif).pvalue:.2g}")

LightGBM optimizado:  0.8011 ± 0.0081
LightGBM original:    0.7973 ± 0.0078
Regresión logística:  0.7645 ± 0.0091
Optimizado vs. LightGBM original: gana en 24 de 25 folds · Wilcoxon p = 1.8e-07
Optimizado vs. Regresión logística: gana en 25 de 25 folds · Wilcoxon p = 6e-08


**Conclusión:**
- La optimización con Optuna **sí aporta mejora real**: LightGBM optimizado supera de forma consistente y estadísticamente significativa ($p < 0,05$) tanto a su versión original del Producto 1 como a la regresión logística en los 25 folds.
- Elegimos **LightGBM optimizado** como nuestro modelo candidato principal.